# Hypothesis Testing Explorer — scipy + PySpark

Hypothesis testing answers a simple question: is the difference I observe in my data real, or could it be due to random chance? The answer is a p-value — a number between 0 and 1. If it is below a threshold (usually 0.05), we say the difference is statistically significant. This project builds an interactive tool that runs five common hypothesis tests, checks their assumptions, and translates the output into plain English.

For large datasets, the data preparation routes through PySpark — demonstrating that the same statistical question can be answered at any scale by choosing the right computational tool.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sys, os
sys.path.insert(0, os.path.abspath('.'))

from config import ALPHA, PYSPARK_THRESHOLD, NORMALITY_THRESHOLD, LARGE_SAMPLE_NORMALITY
from stats_engine import run_test, check_normality, check_equal_variance

print(f'Significance threshold (alpha): {ALPHA}')
print(f'PySpark routing threshold: {PYSPARK_THRESHOLD:,} rows')
print(f'Shapiro-Wilk skipped above: {LARGE_SAMPLE_NORMALITY:,} rows')

## The Five Tests — When to Use Each

| Test | Use when | Parametric | Assumes equal variance |
|------|----------|------------|-----------------------|
| Independent t-test | Two separate groups, continuous, normal | Yes | Yes |
| Welch's t-test | Two separate groups, continuous, unequal variance | Yes | No |
| Paired t-test | Same subjects measured twice | Yes | N/A |
| Mann-Whitney U | Non-normal data or ordinal data | No | No |
| Chi-square | Categorical variables, frequency data | No | N/A |

**Parametric** tests assume the data follows a specific distribution (usually normal). **Non-parametric** tests make no such assumption — they work by ranking the data instead of using the raw values. Non-parametric tests are more robust but have slightly less statistical power when the parametric assumptions are actually met.

## P-values Explained Simply

A p-value of 0.03 means: if there were truly no difference between the groups, we would only see a result this extreme 3% of the time by random chance. Since 3% is below our threshold of 5%, we conclude the difference is real.

**A p-value is NOT the probability that the null hypothesis is true.** It is the probability of seeing data at least this extreme, *assuming the null hypothesis is true*. This distinction is important and frequently misunderstood.

Practical interpretation:
- `p = 0.001` → extremely unlikely to be random chance; strong evidence of a difference
- `p = 0.04` → below the 0.05 threshold; we reject the null hypothesis
- `p = 0.06` → just above the threshold; we do not reject (but this is not evidence of *no* difference)
- `p = 0.80` → very likely to be random chance; no evidence of a difference

The threshold of 0.05 is a convention, not a law of nature. Some fields use 0.01 (stricter) or 0.10 (more lenient).

In [ ]:
# Demonstrate all five tests on generated example data
rng = np.random.default_rng(42)

# Two groups with different means
group_a = rng.normal(50, 10, 80).tolist()
group_b = rng.normal(57, 10, 80).tolist()

# Paired data (same subjects, before/after)
before = rng.normal(140, 15, 60).tolist()
after = (np.array(before) - rng.normal(8, 4, 60)).tolist()

# Categorical data
observed_counts = [45, 30, 25]
expected_counts = [33, 33, 34]

tests_to_run = [
    ('independent_ttest', group_a, group_b),
    ('welch_ttest',       group_a, group_b),
    ('paired_ttest',      before,  after),
    ('mannwhitney',       group_a, group_b),
    ('chisquare',         observed_counts, expected_counts),
]

for test_name, g1, g2 in tests_to_run:
    result = run_test(test_name, g1, g2)
    print(f'\n=== {test_name.upper()} ===')
    print(f'  Verdict: {result["verdict"]}')
    print(f'  Statistic: {result["statistic"]:.4f}  |  p-value: {result["p_value"]:.4f}')
    print(f'  Engine: {result["engine"]}')

## Assumption Checking

Every parametric test has assumptions. Running a t-test on non-normal data or data with very unequal variances produces unreliable results — the p-value may be wrong. The app checks normality (Shapiro-Wilk) and equal variance (Levene's test) before running parametric tests and recommends alternatives if assumptions are violated. This is what a statistician does before reporting results — and what a good data tool should automate.

**Shapiro-Wilk normality test**
- Null hypothesis: the data is normally distributed
- If p < 0.05, we reject normality — the data is likely not normal
- Limitation: unreliable on very large samples (n > 5,000) — it detects trivial deviations that are statistically significant but practically irrelevant

**Levene's test for equal variance**
- Null hypothesis: the two groups have equal variances
- If p < 0.05, the variances differ significantly — use Welch's t-test instead of the standard t-test

In [ ]:
# Show assumption checking in action
# Generate clearly non-normal data (exponential distribution)
rng2 = np.random.default_rng(99)
non_normal_a = rng2.exponential(scale=5, size=80).tolist()
non_normal_b = rng2.exponential(scale=8, size=80).tolist()

print('=== Running independent t-test on non-normal (exponential) data ===')
result_nn = run_test('independent_ttest', non_normal_a, non_normal_b)

print(f'\nVerdict: {result_nn["verdict"]}')
print(f'p-value: {result_nn["p_value"]:.4f}')

print('\n--- Assumption checks ---')
for check_name, check_val in result_nn['assumption_checks'].items():
    print(f'  {check_name}: {check_val["interpretation"]}')

if result_nn['recommendation']:
    print(f'\nRECOMMENDATION: {result_nn["recommendation"]}')

print('\n=== Running Mann-Whitney U on the same data (correct choice) ===')
result_mw = run_test('mannwhitney', non_normal_a, non_normal_b)
print(f'Verdict: {result_mw["verdict"]}')
print(f'p-value: {result_mw["p_value"]:.4f}')

## PySpark for Large Datasets

When a dataset has hundreds of thousands of rows, pandas loads it all into memory on a single core. PySpark distributes the computation across all available cores (`local[*]` mode) and — at production scale — across a cluster. The statistical question is identical. The computational path is different. This app switches automatically at 50,000 rows. Below that threshold, Spark's initialisation overhead (~3 seconds) is not justified. Above it, the parallelism pays off.

**The honest limitation:** The actual test statistic is computed by scipy on a representative sample — PySpark handles the data preparation and descriptive statistics. This is a common production pattern: distributed computing for data scale, established libraries for statistical computation.

In an interview: *"Spark doesn't have native hypothesis testing in the way scipy does — the production pattern is distributed data prep plus established statistical libraries for the test itself. I implemented exactly that split."*

In [ ]:
# Demonstrate the PySpark path by generating a 100,000-row synthetic dataset
import tempfile, os, csv

rng3 = np.random.default_rng(7)
large_group1 = rng3.normal(100, 20, 60_000)
large_group2 = rng3.normal(104, 20, 60_000)

print(f'Group 1: {len(large_group1):,} rows  |  mean = {large_group1.mean():.2f}')
print(f'Group 2: {len(large_group2):,} rows  |  mean = {large_group2.mean():.2f}')
print(f'Total: {len(large_group1)+len(large_group2):,} rows  →  routes to PySpark (threshold = {PYSPARK_THRESHOLD:,})')

# Write to temp CSV files
tmp1 = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='')
tmp2 = tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False, newline='')

writer1 = csv.writer(tmp1); writer1.writerow(['value'])
for v in large_group1: writer1.writerow([round(float(v), 4)])
tmp1.close()

writer2 = csv.writer(tmp2); writer2.writerow(['value'])
for v in large_group2: writer2.writerow([round(float(v), 4)])
tmp2.close()

print(f'\nCSV files written: {tmp1.name}, {tmp2.name}')

from spark_engine import run_test_spark

print('\nRunning via PySpark (this takes ~5s to initialise Spark)...')
spark_result = run_test_spark(
    test_name='independent_ttest',
    group1_path=tmp1.name,
    group2_path=tmp2.name,
    col1='value',
    col2='value',
)

print(f'\nEngine: {spark_result["engine"]}')
print(f'Total rows processed: {spark_result["total_rows"]:,}')
print(f'Sampled for test statistic: {spark_result["sampled_for_test"]:,}')
print(f'Verdict: {spark_result["verdict"]}')
print(f'p-value: {spark_result["p_value"]:.6f}')
print(f'\nSpark-computed stats (group 1): {spark_result["spark_stats"]["group1"]}')
print(f'Spark-computed stats (group 2): {spark_result["spark_stats"]["group2"]}')

# Compare with small-data path on first 1,000 rows
small_result = run_test('independent_ttest', large_group1[:1000].tolist(), large_group2[:1000].tolist())
print(f'\n--- Engine comparison ---')
print(f'PySpark + scipy (120k rows): engine = "{spark_result["engine"]}"')
print(f'pandas + scipy   (2k rows): engine = "{small_result["engine"]}"')

os.unlink(tmp1.name)
os.unlink(tmp2.name)

## When to Use Each Tool

| Concern | pandas + scipy | PySpark + scipy |
|---------|---------------|----------------|
| Dataset size | Up to ~1M rows in memory | Unlimited (distributed) |
| Speed on small data | Fast | Slow (3-5s init overhead) |
| Speed on large data | Slow (single core) | Fast (all cores / cluster) |
| Setup complexity | None | Requires Java + Spark |
| Production pattern | Single-machine analysis | Enterprise data pipelines |

The 50,000-row threshold is the crossover point where Spark's initialisation overhead is justified by the parallelism gains. Below it, pandas is faster. Above it, Spark wins — and at millions of rows, there is no contest.

## Azure App Service Deployment

**Key requirement: Java must be installed for PySpark.** PySpark requires the Java Virtual Machine. The Dockerfile installs OpenJDK via `apt-get` before installing Python packages. Without Java, Spark fails to initialise with a cryptic error. This adds ~200MB to the Docker image but is unavoidable. On Azure App Service, the container starts cleanly because Java is installed at build time, not at runtime.

### Deployment command sequence

```bash
# 1. Create resource group
az group create --name hypothesis-testing-rg --location westeurope

# 2. Create App Service plan (B1 Linux)
az appservice plan create \
  --name hypothesis-testing-plan \
  --resource-group hypothesis-testing-rg \
  --sku B1 --is-linux
# Scale to F1 via portal after creation

# 3. Create web app
az webapp create \
  --name hypothesis-testing-xoc \
  --resource-group hypothesis-testing-rg \
  --plan hypothesis-testing-plan \
  --runtime "PYTHON:3.11"

# 4. Set startup command
az webapp config set \
  --name hypothesis-testing-xoc \
  --resource-group hypothesis-testing-rg \
  --startup-file "gunicorn main:app --workers 1 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000 --timeout 600"

# 5. Enable build during deployment
az webapp config appsettings set \
  --name hypothesis-testing-xoc \
  --resource-group hypothesis-testing-rg \
  --settings SCM_DO_BUILD_DURING_DEPLOYMENT=true

# 6. Package and deploy
cd hypothesis-testing-explorer && zip -r deploy.zip . \
  -x "*.git*" -x "venv/*" -x "__pycache__/*" \
  -x "*.ipynb_checkpoints*" -x "uploads/*" -x "plots/*"

az webapp deployment source config-zip \
  --name hypothesis-testing-xoc \
  --resource-group hypothesis-testing-rg \
  --src deploy.zip
```

## Key Findings

TBD — populate after running the live demo against the three built-in example datasets.

Include:
- Exam scores (Method A vs B): verdict and p-value
- Sales by region (North vs South): verdict and p-value  
- Blood pressure (before vs after): verdict and p-value
- Any assumption violations detected and recommendations given